# Ministral-8B-Instruct-2410 FineTuning — NL to ASP Translation

In [ ]:
from huggingface_hub import login
login('YOUR HUGGINGFACE_TOKEN')

In [ ]:
import os
import json
from pathlib import Path
from datasets import load_dataset
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from transformers import EarlyStoppingCallback
import trl
from trl import SFTTrainer, SFTConfig
import torch
import peft
from peft import LoraConfig


print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Torch:", torch.__version__)


## Dataset Generation

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert in Translating the Natural language (NL) into "
    "Answer Set Programming (ASP) translation. "
    "Always provide precise, syntactically and semantically correct translations of NL into ASP."
)

def load_data(path, test_size=0.1, seed=42):
    """
    Load JSON dataset and convert to conversational format.
    Expects: {"data_dict": [{"NL_V2": ..., "ASP": ...}, ...]}
    Format:
        <s>[INST]You are an expert in NL→ASP translation...

        Translate the following natural language...[/INST]{completion}</s>
    """
    def create_conversation(sample):
        return {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        f"{SYSTEM_PROMPT}\n\n"                          # ← \n\n separator
                        f"Translate the following natural language to Answer Set Programming:\n\n"  # ← \n\n separator
                        f"natural language: {sample['NL_V2']}"
                    )
                },
                {
                    "role": "assistant",
                    "content": sample["ASP"]
                },
            ]
        }

    dataset = load_dataset("json", data_files=path, field="data_dict", split="train")
    dataset = dataset.map(create_conversation, remove_columns=dataset.features, batched=False)
    print("Dataset converted to conversational format.")
    print(f"Total samples: {len(dataset)}")
    print("\nExample message content:")
    print(dataset[0]["messages"][0]["content"])   # print content to verify separator

    if test_size == 0:
        return dataset, None

    split_dataset = dataset.train_test_split(test_size=test_size, seed=seed)
    train_dataset = split_dataset["train"].shuffle(seed=seed)
    test_dataset  = split_dataset["test"]

    return train_dataset, test_dataset

In [ ]:
dataset_file = "Path/To/Your/train_data.json" 
train_data, test_data = load_data(dataset_file, test_size=0.1)

# Optional: inspect an example
print("Example from train set:")
print(train_data[2])

print(f"Train dataset size: {len(train_data)}")
print(f"Test dataset size:  {len(test_data)}")

## Model Loading

In [ ]:
base_model_name = "mistralai/Ministral-8B-Instruct-2410"

def load_prepare_model(model_name=base_model_name):
    """
    Load Ministral-8B-Instruct-2410 in bfloat16 on ROCm GPU.
    MinistralConfig / MistralForCausalLM — pure text model, clean AutoModel load.
    dtype= is correct for transformers 5.x.
    """
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16,
        device_map="auto",
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print(f"Model loaded:  {model_name}")
    print(f"Model class:   {model.__class__.__name__}")
    print(f"GPU memory:    {torch.cuda.memory_allocated() / 1e9:.2f} GB")

    return model, tokenizer

transformers version: 4.57.3


In [ ]:
base_model = base_model_name
model, tokenizer = load_prepare_model(model_name=base_model)

## Preprocessing — Completion-Only Format

In [ ]:
def preprocess_to_completion_format(dataset, tokenizer):
    """
    Convert messages to prompt+completion format for SFTConfig(completion_only_loss=True).
    Loss computed ONLY on the CNL completion tokens — system/user turns are masked.

    Ministral-8B-Instruct-2410 format:
        <s>[INST] {system}\n\n{user} [/INST]{completion}</s>

    Split point: '[/INST]'
    """
    ASSISTANT_HEADER = "[/INST]"

    def convert(sample):
        full_text = tokenizer.apply_chat_template(
            sample["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )

        split_idx = full_text.rfind(ASSISTANT_HEADER)
        if split_idx == -1:
            raise ValueError(
                f"'[/INST]' not found in template output.\n"
                f"Text: {full_text[:300]}\n"
                f"Check that tokenizer is Ministral-8B-Instruct-2410."
            )

        prompt     = full_text[:split_idx + len(ASSISTANT_HEADER)]
        completion = full_text[split_idx + len(ASSISTANT_HEADER):]

        return {"prompt": prompt, "completion": completion}

    return dataset.map(convert, remove_columns=dataset.column_names)


train_data_processed = preprocess_to_completion_format(train_data, tokenizer)
test_data_processed  = preprocess_to_completion_format(test_data,  tokenizer)

# Verify the split is correct
sample = train_data_processed[0]
print("--- PROMPT (masked from loss) ---")
print(sample["prompt"])
print()
print("--- COMPLETION (loss computed here) ---")
print(sample["completion"])

## Trainer Setup & Training

In [ ]:

max_seq_length = 1024

def create_trainer(model, tokenizer, train_dataset, eval_dataset=None,
                   max_seq_length=max_seq_length, num_epochs=15, early_stopping_patience=3):
    """
    SFTTrainer configured for Ministral-8B-Instruct-2410 on ROCm.
    - completion_only_loss=True: loss on CNL output tokens only
    """
    peft_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.15,
        use_rslora=True,
        use_dora=False,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        task_type="CAUSAL_LM",
    )

    training_arguments = SFTConfig(
        output_dir="PATH/TO/SAVE/ADAPTER",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=1,
        optim="adamw_torch",
        logging_steps=25,
        learning_rate=5e-5,
        weight_decay=0.05,
        fp16=False,
        bf16=True,
        max_grad_norm=0.3,
        max_steps=-1,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        neftune_noise_alpha=5.0,
        load_best_model_at_end=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        max_length=max_seq_length,
        completion_only_loss=True,
        dataset_text_field=None,
        report_to="none",
    )

    callbacks = []
    if eval_dataset is not None and early_stopping_patience > 0:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=early_stopping_patience))

    trainer = SFTTrainer(
        model=model,
        args=training_arguments,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
        callbacks=callbacks if callbacks else None,
    )

    return trainer

In [ ]:
trainer = create_trainer(
    model, tokenizer,
    train_dataset=train_data_processed,
    eval_dataset=test_data_processed,
    num_epochs=15,
    early_stopping_patience=3
)

trainer.train()

print("Best checkpoint:", trainer.state.best_model_checkpoint)

# Save final adapter
trainer.save_model("PATH/TO/SAVE/ADAPTER")
tokenizer.save_pretrained("PATH/TO/SAVE/ADAPTER")
print("PATH/TO/SAVE/ADAPTER")

In [ ]:
trainer.state.best_model_checkpoint  ## path to best checkpoint SAVED